In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Paths -- edit these for your environment.
RELEASE_DIR = "/PATH/TO/release"             # de-identified, date-shifted release files
SHIFT_MAP   = "/PATH/TO/date_shift_map.csv"  # written by microbiology_cultures_cohort_query.ipynb

In [ ]:
file_names = ['microbiology_cultures_cohort.csv',
              'microbiology_cultures_adi_scores.csv',
              'microbiology_cultures_antibiotic_class_exposure_augmented.csv',
              'microbiology_cultures_antibiotic_subtype_exposure_augmented.csv',
              'microbiology_cultures_comorbidity.csv',
              'microbiology_cultures_demographics.csv',
              'microbiology_cultures_microbial_resistance_augmented.csv',
              'microbiology_cultures_nursing_home_visits_augmented.csv',
              'microbiology_cultures_prior_infecting_organism_augmented.csv',
              'microbiology_cultures_prior_med_augmented.csv',
              'microbiology_cultures_priorprocedures.csv',
              'microbiology_cultures_ward_info.csv']

In [ ]:
offset_map = pd.read_csv(SHIFT_MAP) 
offset_map['date_offset'] = pd.to_timedelta(offset_map['date_offset'])
# offset_map['date_offset'] = pd.to_timedelta(offset_map['date_offset'].astype(int), unit='D')

for file in file_names:
    data = pd.read_csv(f"{RELEASE_DIR}/dedup/{file}")
    # Check for Columns that is in the list provided
    date_columns = [col for col in data.columns if col in ['order_time_jittered_utc','medicationOrderDTS','visit_date','recorded_resistsant_time','prior_infecting_organism_recorded_time']]
    print(date_columns)
    data = data.merge(offset_map, on='anon_id', how='left')
    # Ensure datetime format
    data[date_columns] = data[date_columns].apply(pd.to_datetime, errors='coerce')

    # Apply shift using the merged date_offset
    for col in date_columns:
        data[col + '_shifted'] = data[col] + data['date_offset']
        # Only Keep the date information
        data[col + '_shifted'] = data[col + '_shifted'].dt.date
    # Drop the original date columns
    data.drop(columns=date_columns, inplace=True)
    # Drop the date_offset column
    data.drop(columns=['date_offset'], inplace=True)

    # save file
    data.to_csv(f"{RELEASE_DIR}/deid/{file}", index=False)

In [ ]:
import glob
import os

input_dir = f"{RELEASE_DIR}/dedup"
csv_files = glob.glob(os.path.join(input_dir, "*.csv"))

output_dir = f"{RELEASE_DIR}/time_deid_second_pass"

offset_map = pd.read_csv(SHIFT_MAP) 
offset_map['date_offset'] = pd.to_timedelta(offset_map['date_offset'])

print(f"Found {len(csv_files)} files.")

for file in csv_files:
    print(f"Processing {os.path.basename(file)}")

    data = pd.read_csv(file)

    # find *_shifted columns
    shifted_cols = [c for c in data.columns if c.endswith("_shifted")]
    print("Shifted columns:", shifted_cols)

    # merge new offsets
    data = data.merge(offset_map, on="anon_id", how="left")

    # convert _shifted columns to datetime (they are currently date strings)
    for col in shifted_cols:
        data[col] = pd.to_datetime(data[col], errors="coerce")

    # apply new shift (overwrite existing)
    for col in shifted_cols:
        data[col] = (data[col] + data["date_offset"]).dt.date

    # remove offset column
    data.drop(columns=["date_offset"], inplace=True)

    # save into second pass folder
    output_path = os.path.join(output_dir, os.path.basename(file))
    data.to_csv(output_path, index=False)